In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"C:\Users\dinan\OneDrive\Desktop\airtel-telecom-analytics\data\raw\airtel.db")
def q(sql):
    return pd.read_sql(sql, conn)


In [4]:
q("""
SELECT COUNT(*)
FROM customer_features;
""")

,COUNT(*)
0,22000


In [5]:
q("""
SELECT ROUND(AVG(actual_arpu_6m), 2)
FROM customer_features;
""")

,"ROUND(AVG(actual_arpu_6m), 2)"
0,344.46


In [6]:
q("""
SELECT SUM(complaint_count = 0)
FROM customer_features;
""")

,SUM(complaint_count = 0)
0,6631


In [7]:
q("""
SELECT ab_group, COUNT(*)
FROM customer_features
GROUP BY ab_group;
""")

,ab_group,COUNT(*)
0,A,10996
1,B,11004


In [8]:
q("""
SELECT connection_type,
       COUNT(*) AS subs,
       ROUND(100.0*COUNT(*)/22000, 2) AS pct
FROM customer_features
GROUP BY connection_type
ORDER BY subs DESC;
""")

,connection_type,subs,pct
0,Prepaid,17149,77.95
1,Postpaid,4851,22.05


# Airtel SQL Data Analysis — Areas A to F


# AREA A — THE SUBSCRIBER BASE

### A1 — Subscribers by connection 

 How many subscribers are prepaid and how many are postpaid? Calculate each group's percentage of the total base.

In [9]:
q("""
SELECT connection_type,                         -- prepaid or postpaid
       COUNT(*) AS subs,                         -- subscribers in each group
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 2) AS pct -- share of total base
FROM customer_features                          -- one row per subscriber
GROUP BY connection_type                        -- separate connection types
ORDER BY subs DESC;                             -- largest group first
""")

,connection_type,subs,pct
0,Prepaid,17149,77.95
1,Postpaid,4851,22.05


### A2 — Five most popular plans
Which five plans have the most subscribers? Show plan name, plan type, price and subscriber count.

In [10]:
q("""
SELECT plan_name, plan_type, price_inr,              -- plan details
       COUNT(*) AS subs                              -- subscribers on each plan
FROM customer_features                              -- one row per subscriber
GROUP BY plan_name, plan_type, price_inr             -- one row per plan
ORDER BY subs DESC                                   -- most popular first
LIMIT 5;                                             -- return five plans
""")

,plan_name,plan_type,price_inr,subs
0,Prepaid Annual 1799,Prepaid,1799,2901
1,Prepaid Quarterly 719,Prepaid,719,2561
2,Prepaid Max 479,Prepaid,479,2392
3,Prepaid Entertainment 399,Prepaid,399,2142
4,Prepaid Data 349,Prepaid,349,1884


### A3 — Five circles with most subscribers 
Which five circles have the most subscribers? Include region and subscriber count.

In [11]:
q("""
SELECT circle_name, region,                          -- circle and region
       COUNT(*) AS subs                              -- subscribers in circle
FROM customer_features
GROUP BY circle_name, region                         -- one row per circle/region
ORDER BY subs DESC
LIMIT 5;
""")

,circle_name,region,subs
0,Maharashtra,West,1428
1,Andhra Pradesh,South,1412
2,Mumbai,West,1395
3,Karnataka,South,1361
4,North East,East,1318


### A4 — Subscriber count by age band
How many subscribers are in each age band? Show count and percentage, ordered by age band.

In [12]:
q("""
SELECT age_band,                                 -- age category
       COUNT(*) AS subs,                         -- subscribers in band
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 1) AS pct -- percentage of base
FROM customer_features
GROUP BY age_band
ORDER BY age_band;                               -- natural band order
""")

,age_band,subs,pct
0,18-24,5268,23.9
1,25-34,7467,33.9
2,35-44,4824,21.9
3,45-54,2895,13.2
4,55+,1546,7.0


### A5 — Acquisition channel
How many subscribers came from each acquisition channel? Show count and percentage, highest first.

In [13]:
q("""
SELECT acquisition_channel,                     -- source of acquisition
       COUNT(*) AS subs,                         -- subscribers from channel
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 1) AS pct -- share of total base
FROM customer_features
GROUP BY acquisition_channel
ORDER BY subs DESC;
""")

,acquisition_channel,subs,pct
0,Retailer,9287,42.2
1,Airtel App,6513,29.6
2,Website,3081,14.0
3,Call Center,1770,8.0
4,Doorstep,1349,6.1


### A6 — Device type and operating system
How many subscribers use each device type and operating system combination? Count each combination.

In [14]:
q("""
SELECT device_type, os,                          -- device and OS combination
       COUNT(*) AS subs                          -- subscribers in combination
FROM customer_features
GROUP BY device_type, os                         -- one row per combination
ORDER BY subs DESC;
""")

,device_type,os,subs
0,Smartphone,Android,15613
1,Smartphone,iOS,3358
2,Feature Phone,Other,3029


### A7 — Subscriber tenure distribution
 How many subscribers have 0, 1, 2 and 3 completed years of tenure? Show count and percentage.

In [15]:
q("""
SELECT tenure_years,                             -- completed years with company
       COUNT(*) AS subs,                         -- subscribers at each tenure
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 1) AS pct -- percentage of base
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,subs,pct
0,0,4610,21.0
1,1,6961,31.6
2,2,7055,32.1
3,3,3374,15.3


# AREA B — REVENUE AND ARPU

### B1 — Average stated ARPU
 Calculate the average stated_arpu across subscribers. Also show the number of subscribers included.

In [16]:
q("""
SELECT ROUND(AVG(stated_arpu), 2) AS avg_arpu,     -- average stated plan price
       COUNT(*) AS subs                            -- number of subscribers included
FROM customer_features;
""")

,avg_arpu,subs
0,421.65,22000


### B2 — Average actual ARPU and monthly total
Calculate the average actual_arpu_6m and the total monthly revenue represented by this measure.

In [17]:
q("""
SELECT ROUND(AVG(actual_arpu_6m), 2) AS avg_actual, -- average actual ARPU
       ROUND(SUM(actual_arpu_6m), 0) AS monthly_total -- total monthly revenue represented
FROM customer_features;
""")

,avg_actual,monthly_total
0,344.46,7578210.0


### B3 — Difference between stated and actual ARPU
Compare stated_arpu with actual_arpu_6m. Calculate the average gap and percentage difference. State the denominator used.

In [18]:
q("""
SELECT ROUND(AVG(stated_arpu), 2) AS stated,       -- average stated amount
       ROUND(AVG(actual_arpu_6m), 2) AS actual,    -- average received amount
       ROUND(AVG(arpu_gap), 2) AS gap,             -- average difference
       ROUND(100.0 * (SUM(stated_arpu) - SUM(actual_arpu_6m))
             / NULLIF(SUM(actual_arpu_6m), 0), 2) AS over_pct -- gap as % of actual revenue
FROM customer_features;
""")

,stated,actual,gap,over_pct
0,421.65,344.46,77.18,22.41


### B4 — Stated and actual ARPU by connection type
 Compare stated and actual ARPU for prepaid and postpaid subscribers separately.

In [19]:
q("""
SELECT connection_type,                            -- prepaid or postpaid
       COUNT(*) AS subs,                            -- group size
       ROUND(AVG(stated_arpu), 2) AS stated,        -- plan-price ARPU
       ROUND(AVG(actual_arpu_6m), 2) AS actual      -- actual ARPU measure
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,subs,stated,actual
0,Postpaid,4851,959.65,970.42
1,Prepaid,17149,269.46,167.40


### B5 — Five plans with highest actual monthly revenue
 Find the five plans with the highest actual monthly revenue. Show subscriber count, total revenue and revenue per subscriber.

In [20]:
q("""
SELECT plan_name,                                   -- plan name
       COUNT(*) AS subs,                            -- subscribers on plan
       ROUND(SUM(actual_arpu_6m), 0) AS rev,        -- total monthly revenue
       ROUND(AVG(actual_arpu_6m), 2) AS per_sub     -- revenue per subscriber
FROM customer_features
GROUP BY plan_name
ORDER BY rev DESC
LIMIT 5;
""")

,plan_name,subs,rev,per_sub
0,Postpaid Infinity 1499,1431,2171756.0,1517.65
1,Postpaid Family 999,1239,1249246.0,1008.27
2,Postpaid 749,946,718959.0,760.00
3,Prepaid Entertainment 399,2142,671850.0,313.66
4,Prepaid Data 349,1884,520066.0,276.04


### B6 — Five circles with highest actual monthly revenue
Find the five circles with the highest actual monthly revenue. Show subscriber count and total revenue.

In [21]:
q("""
SELECT circle_name,                                 -- circle name
       COUNT(*) AS subs,                            -- subscribers in circle
       ROUND(SUM(actual_arpu_6m), 0) AS rev         -- total actual revenue
FROM customer_features
GROUP BY circle_name
ORDER BY rev DESC
LIMIT 5;
""")

,circle_name,subs,rev
0,Mumbai,1395,473288.0
1,Karnataka,1361,468515.0
2,Andhra Pradesh,1412,463370.0
3,North East,1318,460998.0
4,Maharashtra,1428,450032.0


### B7 — Payments and amount received January–June 2025
 Calculate the number of payment transactions and total amount received from 1 January to 30 June 2025 using recharges.

In [22]:
q("""
SELECT COUNT(*) AS payments,                     -- number of transactions
       ROUND(SUM(amount_inr) / 10000000.0, 2) AS crore_inr -- amount in crore INR
FROM recharges                                   -- one row per payment
WHERE recharge_date >= '2025-01-01'              -- inclusive start date
  AND recharge_date < '2025-07-01';              -- exclusive end date includes all of June
""")

,payments,crore_inr
0,83693,4.55


### B8 — Actual ARPU by tenure
 Compare average actual_arpu_6m for subscribers with 0, 1, 2 and 3 years of tenure.

In [23]:
q("""
SELECT tenure_years,                                -- completed years
       COUNT(*) AS subs,                            -- subscribers in group
       ROUND(AVG(actual_arpu_6m), 2) AS arpu        -- average actual ARPU
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,subs,arpu
0,0,4610,329.10
1,1,6961,342.72
2,2,7055,350.13
3,3,3374,357.22


### B9 — Subscribers with no recharge and churn
Count subscribers with recharge_count = 0, calculate the churn percentage for that group, and inspect stated_arpu carefully. Explain why it may be misleading.

In [24]:
q("""
SELECT SUM(CASE WHEN recharge_count = 0 THEN 1 ELSE 0 END) AS paid_nothing, -- zero-recharge subscribers
       ROUND(100.0 * AVG(CASE WHEN recharge_count = 0 THEN is_churned END), 2) AS their_churn, -- churn within group
       ROUND(AVG(CASE WHEN recharge_count = 0 THEN stated_arpu END), 2) AS zero_recharge_stated_arpu -- stated ARPU in that group
FROM customer_features;
""")

,paid_nothing,their_churn,zero_recharge_stated_arpu
0,3711,44.3,186.2


# AREA C — RECHARGE BEHAVIOUR

### C1 — Payment count and distinct subscribers by transaction type
For each txn_type, count payment transactions and distinct subscribers.

In [25]:
q("""
SELECT txn_type,                                   -- transaction type
       COUNT(*) AS payments,                       -- number of payment rows
       COUNT(DISTINCT customer_id) AS subs         -- unique subscribers
FROM recharges
GROUP BY txn_type;
""")

,txn_type,payments,subs
0,Bill Payment,108419,4851
1,Recharge,299670,17149


### C2 — Overall average payment
Calculate the average amount_inr across all rows in recharges. Record the result and explain what population it averages.

In [26]:
q("""
SELECT ROUND(AVG(amount_inr), 2) AS avg_payment, -- average across all payment rows
       COUNT(*) AS payment_rows                    -- denominator: payment transactions
FROM recharges;
""")

,avg_payment,payment_rows
0,532.59,408089


### C3 — Average, minimum and maximum by transaction type
 Calculate average, minimum and maximum payment amount separately for each txn_type.

In [27]:
q("""
SELECT txn_type,                                   -- payment category
       ROUND(AVG(amount_inr), 2) AS avg_amt,       -- average payment
       MIN(amount_inr) AS mn,                     -- smallest payment
       MAX(amount_inr) AS mx                      -- largest payment
FROM recharges
GROUP BY txn_type;
""")

,txn_type,avg_amt,mn,mx
0,Bill Payment,1082.42,399,1873
1,Recharge,333.66,99,1799


### C4 — Payment frequency and total by connection type
For prepaid and postpaid subscribers, calculate average recharge_count and average recharge_total_6m.

In [28]:
q("""
SELECT connection_type,                           -- prepaid or postpaid
       ROUND(AVG(recharge_count), 2) AS avg_payments, -- average number of payments
       ROUND(AVG(recharge_total_6m), 0) AS avg_total_6m -- average six-month total
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,avg_payments,avg_total_6m
0,Postpaid,5.39,5823.0
1,Prepaid,3.36,1004.0


### C5 — Payment channels by transaction count
Find the payment channels with the highest number of transactions. Show count and percentage of all payments.

In [29]:
q("""
SELECT channel,                                   -- payment channel
       COUNT(*) AS payments,                      -- transactions on channel
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM recharges), 1) AS pct -- share of all payments
FROM recharges
GROUP BY channel
ORDER BY payments DESC;
""")

,channel,payments,pct
0,Website,111163,27.2
1,Airtel App,111083,27.2
2,USSD,75257,18.4
3,Retailer,74406,18.2
4,AutoPay,36180,8.9


### C6 — Channel share within each transaction type
Compare payment-channel usage for prepaid and postpaid transactions. Calculate each channel's percentage within its own txn_type group.

In [30]:
q("""
SELECT txn_type, channel,                         -- transaction type and channel
       COUNT(*) AS n,                             -- number of transactions
       ROUND(100.0 * COUNT(*) /
             SUM(COUNT(*)) OVER (PARTITION BY txn_type), 1) AS pct -- share within txn_type
FROM recharges
GROUP BY txn_type, channel
ORDER BY txn_type, n DESC;
""")

,txn_type,channel,n,pct
0,Bill Payment,Airtel App,36186,33.4
1,Bill Payment,AutoPay,36180,33.4
2,Bill Payment,Website,36053,33.3
3,Recharge,USSD,75257,25.1
4,Recharge,Website,75110,25.1
5,Recharge,Airtel App,74897,25.0
6,Recharge,Retailer,74406,24.8


### C7 — Churn by subscriber recharge count
Group subscribers by recharge_count and calculate subscriber count and churn percentage for each count. Restrict the displayed range to counts of eight or fewer and comment on small groups.

In [31]:
q("""
SELECT recharge_count,                            -- number of payments
       COUNT(*) AS subs,                          -- subscribers at this count
       ROUND(100.0 * AVG(is_churned), 2) AS churn -- churn percentage
FROM customer_features
WHERE recharge_count <= 8                         -- requested range
GROUP BY recharge_count
ORDER BY recharge_count;
""")

,recharge_count,subs,churn
0,0,3711,44.30
1,1,2762,40.04
2,2,2339,28.35
3,3,1026,54.09
4,4,714,76.05
5,5,3407,13.77
6,6,6732,1.52
7,7,507,5.52
8,8,32,90.63


# AREA D — DAILY USAGE OVER TIME

### D1 — Monthly network traffic and active subscribers
For each month, calculate total data traffic in TB and the number of distinct active subscribers.

In [32]:
q("""
SELECT month,                                    -- month bucket
       COUNT(DISTINCT customer_id) AS active_subs, -- distinct active subscribers
       ROUND(SUM(total_mb) / 1024.0 / 1024.0, 1) AS total_tb -- MB to TB
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

,month,active_subs,total_tb
0,2025-01,21302,338.0
1,2025-02,21162,307.3
2,2025-03,20366,332.5
3,2025-04,19320,310.3
4,2025-05,18356,363.1
5,2025-06,17326,338.9


### D2 — Average daily data across usage rows
Calculate average data_mb across all rows in usage_daily and show the number of rows used. Explain what the average is divided by.

In [33]:
q("""
SELECT ROUND(AVG(data_mb), 2) AS avg_mb,          -- average per subscriber-day row
       COUNT(*) AS rows_                          -- denominator: subscriber-day rows
FROM usage_daily;
""")

,avg_mb,rows_
0,1572.87,1326739


### D3 — Average per active day and per observed day
From customer_features, calculate average mb_per_active_day and mb_per_observed_day. Compare the measures and explain their denominators.

In [34]:
q("""
SELECT ROUND(AVG(mb_per_active_day), 2) AS per_active_day, -- per day used
       ROUND(AVG(mb_per_observed_day), 2) AS per_observed_day, -- per day observed
       COUNT(*) AS subs                           -- subscriber rows included
FROM customer_features;
""")

,per_active_day,per_observed_day,subs
0,1404.44,524.05,22000


### D4 — Daily data usage by weekday
Compare average daily data_mb across the seven weekdays using strftime('%w', usage_date). Show the number of rows in each group.

In [35]:
q("""
SELECT strftime('%w', usage_date) AS dow,          -- weekday: 0 Sunday to 6 Saturday
       COUNT(*) AS rows_,                          -- subscriber-day rows
       ROUND(AVG(data_mb), 1) AS avg_mb            -- average data on weekday
FROM usage_daily
GROUP BY dow
ORDER BY dow;
""")

,dow,rows_,avg_mb
0,0,190368,1578.1
1,1,189914,1572.9
2,2,183823,1569.5
3,3,191088,1564.5
4,4,190764,1580.0
5,5,190542,1569.5
6,6,190240,1575.5


### D5 — Active subscribers by month
Count distinct active subscribers in each month and describe the trend.

In [36]:
q("""
SELECT month,                                    -- month bucket
       COUNT(DISTINCT customer_id) AS active_subs -- unique active subscribers
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

,month,active_subs
0,2025-01,21302
1,2025-02,21162
2,2025-03,20366
3,2025-04,19320
4,2025-05,18356
5,2025-06,17326


### D6 — Average voice minutes and SMS by month
Calculate average voice_minutes and sms_count for each month from January to June 2025.

In [37]:
q("""
SELECT strftime('%Y-%m', usage_date) AS m,       -- extract year and month
       ROUND(AVG(voice_minutes), 1) AS avg_voice, -- average voice minutes
       ROUND(AVG(sms_count), 3) AS avg_sms        -- average SMS count
FROM usage_daily
WHERE usage_date >= '2025-01-01' AND usage_date < '2025-07-01' -- Jan–Jun 2025
GROUP BY m
ORDER BY m;
""")

,m,avg_voice,avg_sms
0,2025-01,55.6,0.745
1,2025-02,55.8,0.746
2,2025-03,56.1,0.750
3,2025-04,56.5,0.760
4,2025-05,56.7,0.764
5,2025-06,57.1,0.772


### D7 — Monthly data per subscriber and average active days
For each month, calculate average data use per subscriber in GB and average usage_days.

In [38]:
q("""
SELECT month,                                    -- month bucket
       ROUND(AVG(total_mb) / 1024.0, 2) AS avg_gb_per_sub, -- MB to GB
       ROUND(AVG(usage_days), 1) AS avg_days    -- average active days
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

,month,avg_gb_per_sub,avg_days
0,2025-01,16.25,11.4
1,2025-02,14.87,10.3
2,2025-03,16.72,11.5
3,2025-04,16.44,11.2
4,2025-05,20.26,11.7
5,2025-06,20.03,11.5


### D8 — Monthly data per active day for never-churned subscribers
Restrict the analysis to subscribers who never churned. Join v_usage_month to customer_features and calculate their average data use per active day for each month.

In [39]:
q("""
SELECT m.month,                                  -- month bucket
       ROUND(AVG(m.total_mb * 1.0 / NULLIF(m.usage_days, 0)), 1) AS mb_per_day -- avoid zero division
FROM v_usage_month AS m                          -- one row per subscriber-month
JOIN customer_features AS c                      -- attach churn status
  ON c.customer_id = m.customer_id               -- match subscriber
WHERE c.is_churned = 0                           -- never-churned subscribers
GROUP BY m.month
ORDER BY m.month;
""")

,month,mb_per_day
0,2025-01,1410.6
1,2025-02,1422.1
2,2025-03,1418.1
3,2025-04,1411.8
4,2025-05,1659.5
5,2025-06,1662.2


# AREA E — USAGE INTENSITY AND DATA ALLOWANCE

### E1 — Four quartiles of daily data use
Exclude subscribers with no usage. Divide subscribers into four equal groups using NTILE(4) ordered by mb_per_active_day. Show group size and minimum and maximum daily use.

In [40]:
q("""
WITH n AS (
    SELECT mb_per_active_day AS m,                -- daily data measure
           NTILE(4) OVER (ORDER BY mb_per_active_day) AS q -- four equal groups
    FROM customer_features
    WHERE usage_days > 0                          -- exclude no-usage subscribers
)
SELECT q, COUNT(*) AS subs,                       -- quartile and group size
       ROUND(MIN(m), 0) AS mn,                    -- minimum daily use
       ROUND(MAX(m), 0) AS mx                     -- maximum daily use
FROM n
GROUP BY q
ORDER BY q;
""")

,q,subs,mn,mx
0,1,5499,25.0,657.0
1,2,5499,657.0,1093.0
2,3,5499,1093.0,1572.0
3,4,5499,1572.0,9720.0


### E2 — Churn by daily data-use quartile
Using the same four daily-data groups, calculate churn percentage in each group.

In [41]:
q("""
WITH n AS (
    SELECT is_churned,                            -- carry churn flag
           NTILE(4) OVER (ORDER BY mb_per_active_day) AS q -- daily data quartile
    FROM customer_features
    WHERE usage_days > 0
)
SELECT q, COUNT(*) AS subs,                       -- quartile size
       ROUND(100.0 * AVG(is_churned), 2) AS churn -- churn percentage
FROM n
GROUP BY q
ORDER BY q;
""")

,q,subs,churn
0,1,5499,30.28
1,2,5499,28.11
2,3,5499,22.51
3,4,5499,14.09


### E3 — Voice and data use by connection type and device
 Compare voice minutes per active day and data MB per active day for each connection_type and device_type combination.

In [42]:
q("""
SELECT connection_type, device_type,             -- subscriber segment
       COUNT(*) AS subs,                          -- subscribers in segment
       ROUND(AVG(total_voice_min * 1.0 /
                 NULLIF(usage_days, 0)), 1) AS voice_per_day, -- voice per active day
       ROUND(AVG(mb_per_active_day), 0) AS mb_per_day -- data per active day
FROM customer_features
WHERE usage_days > 0
GROUP BY connection_type, device_type
ORDER BY voice_per_day DESC;
""")

,connection_type,device_type,subs,voice_per_day,mb_per_day
0,Postpaid,Feature Phone,690,48.6,531.0
1,Postpaid,Smartphone,4161,48.0,3353.0
2,Prepaid,Feature Phone,2338,47.9,166.0
3,Prepaid,Smartphone,14807,47.9,1093.0


### E4 — Monthly allowance compared with actual monthly use by plan
For each plan, compare monthly data allowance (monthly_gb) with average actual monthly data use in GB.

In [43]:
q("""
SELECT plan_name, monthly_gb,                    -- plan and monthly allowance
       ROUND(AVG(total_data_mb) / 1024.0 / 6, 2) AS actual_gb_per_month -- six-month data to monthly GB
FROM customer_features
WHERE usage_days > 0
GROUP BY plan_name, monthly_gb
ORDER BY monthly_gb;
""")

,plan_name,monthly_gb,actual_gb_per_month
0,Prepaid Lite 99,15.0,2.88
1,Prepaid Smart 155,30.0,5.72
2,Prepaid Value 209,30.0,5.89
3,Postpaid 399,40.0,7.86
4,Prepaid Max 479,45.0,8.84
5,Prepaid Popular 299,45.0,8.93
6,Prepaid Annual 1799,60.0,11.87
7,Prepaid Quarterly 719,60.0,11.78
8,Postpaid 499,75.0,15.47
9,Prepaid Data 349,75.0,15.13


### E5 — Overall allowance usage and subscribers below half allowance
 Calculate average monthly allowance, average monthly usage, average percentage of allowance used, and number of subscribers using less than half their allowance.

In [44]:
q("""
SELECT ROUND(AVG(monthly_gb), 1) AS allowance_gb, -- average monthly allowance
       ROUND(AVG(total_data_mb / 1024.0 / 6), 2) AS used_gb, -- average monthly use
       ROUND(100.0 * AVG((total_data_mb / 1024.0 / 6) /
                         NULLIF(monthly_gb, 0)), 1) AS pct_of_allowance, -- average subscriber-level share
       SUM(CASE WHEN (total_data_mb / 1024.0 / 6) < monthly_gb * 0.5 THEN 1 ELSE 0 END) AS under_half
FROM customer_features
WHERE usage_days > 0;
""")

,allowance_gb,used_gb,pct_of_allowance,under_half
0,76.8,15.44,19.8,20560


### E6 — Spend, data use and churn by monetary score
 Group subscribers by m_score and compare six-month spend, data per active day and churn.

In [45]:

q("""
WITH scored AS (
    SELECT
        customer_id,
        NTILE(5) OVER (
            ORDER BY recharge_total_6m
        ) AS m_score,
        recharge_total_6m,
        mb_per_active_day,
        is_churned
    FROM customer_features
)
SELECT
    m_score,
    COUNT(*) AS subs,
    ROUND(AVG(recharge_total_6m), 0) AS spend_6m,
    ROUND(AVG(mb_per_active_day), 0) AS mb_per_day,
    ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM scored
GROUP BY m_score
ORDER BY m_score;
""")


,m_score,subs,spend_6m,mb_per_day,churn
0,1,4400,50.0,994.0,53.02
1,2,4400,792.0,902.0,31.11
2,3,4400,1245.0,712.0,13.80
3,4,4400,1986.0,1327.0,11.36
4,5,4400,6261.0,3087.0,9.48


### E7 — Churn by usage-days quartile
Divide subscribers into four equal groups by usage_days. Show each group's usage-day range and churn percentage.

In [46]:
q("""
WITH n AS (
    SELECT usage_days, is_churned,                 -- carry days and churn flag
           NTILE(4) OVER (ORDER BY usage_days) AS q -- four equal groups
    FROM customer_features
)
SELECT q, COUNT(*) AS subs,                       -- group size
       MIN(usage_days) AS mn,                      -- minimum active days
       MAX(usage_days) AS mx,                      -- maximum active days
       ROUND(100.0 * AVG(is_churned), 2) AS churn -- churn percentage
FROM n
GROUP BY q
ORDER BY q;
""")

,q,subs,mn,mx,churn
0,1,5500,0,31,58.56
1,2,5500,31,57,24.45
2,3,5500,57,86,9.49
3,4,5500,86,180,2.51


### E8 — SMS per active day and subscribers who never sent SMS
Calculate average SMS sent per active day, count subscribers who never sent an SMS, and show the population included.

In [47]:
q("""
SELECT ROUND(AVG(total_sms * 1.0 /
                 NULLIF(usage_days, 0)), 3) AS sms_per_active_day, -- SMS per active day
       SUM(CASE WHEN total_sms = 0 THEN 1 ELSE 0 END) AS never_sent_one, -- zero-SMS subscribers
       COUNT(*) AS subs                           -- population included
FROM customer_features
WHERE usage_days > 0;
""")

,sms_per_active_day,never_sent_one,subs
0,0.598,654,21996


# AREA F — ROAMING

### F1 — Roaming and non-roaming days and subscribers
From usage_daily, count roaming and non-roaming days, calculate their percentages, and count distinct subscribers in each group.

In [48]:
q("""
SELECT is_roaming,                               -- 0 non-roaming, 1 roaming
       COUNT(*) AS days,                         -- number of subscriber-days
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM usage_daily), 2) AS pct, -- share of usage days
       COUNT(DISTINCT customer_id) AS subs       -- distinct subscribers
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,days,pct,subs
0,0,1287143,97.02,21994
1,1,39596,2.98,16233


### F2 — Average data and voice on roaming versus non-roaming days
Compare average data_mb and voice_minutes on roaming and non-roaming days.

In [49]:
q("""
SELECT is_roaming,                               -- roaming status
       ROUND(AVG(data_mb), 2) AS avg_mb,         -- average data use
       ROUND(AVG(voice_minutes), 2) AS avg_voice -- average voice minutes
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,avg_mb,avg_voice
0,0,1572.95,56.27
1,1,1570.05,56.09


### F3 — Roaming by prepaid and postpaid subscribers
From customer_features, compare average roaming_days and the percentage who ever roamed for prepaid and postpaid subscribers.

In [50]:
q("""
SELECT connection_type,                          -- prepaid or postpaid
       COUNT(*) AS subs,                          -- subscriber count
       ROUND(AVG(roaming_days), 2) AS avg_roam_days, -- average roaming days
       ROUND(100.0 * AVG(CASE WHEN roaming_days > 0 THEN 1.0 ELSE 0.0 END), 1) AS pct_ever -- ever-roamed share
FROM customer_features
WHERE usage_days > 0
GROUP BY connection_type;
""")

,connection_type,subs,avg_roam_days,pct_ever
0,Postpaid,4851,1.84,74.8
1,Prepaid,17145,1.79,73.5


### F4 — Churn by roaming-day bands
 Create four roaming-day bands (0, 1–2, 3–5 and 6+). Calculate subscriber count and churn percentage for each band. Explain why the pattern may be misleading.

In [51]:
q("""
SELECT CASE                                     -- assign roaming-day band
         WHEN roaming_days = 0 THEN '0'
         WHEN roaming_days <= 2 THEN '1-2'
         WHEN roaming_days <= 5 THEN '3-5'
         ELSE '6+'
       END AS band,
       COUNT(*) AS subs,                         -- subscribers in band
       ROUND(100.0 * AVG(is_churned), 2) AS churn -- churn percentage
FROM customer_features
WHERE usage_days > 0
GROUP BY band
ORDER BY CASE band WHEN '0' THEN 1 WHEN '1-2' THEN 2 WHEN '3-5' THEN 3 ELSE 4 END;
""")

,band,subs,churn
0,0,5763,44.61
1,1-2,9922,21.58
2,3-5,5546,8.69
3,6+,765,3.92


### F5 — Five circles with highest roaming share
For each circle, calculate roaming days as a percentage of total usage days. Show the five highest circles.

In [ ]:
q("""
SELECT circle_name,                              -- circle name
       ROUND(100.0 * SUM(roaming_days) /
             NULLIF(SUM(usage_days), 0), 2) AS pct_roaming -- roaming share of usage days
FROM customer_features
WHERE usage_days > 0
GROUP BY circle_name
ORDER BY pct_roaming DESC
LIMIT 5;


""")

,circle_name,pct_roaming
0,Odisha,3.07
1,Haryana,3.07
2,Delhi,3.07
3,Karnataka,3.05
4,Assam,3.05
